# Text Mining III: word pairs and opinions
## Are these food-court reviews happy or unhappy?
**Salem Othman · Student notebook**

Last week a single word was the unit of everything: count it, weight it, classify with it. Two AI audits caught the same weakness: once *not* is thrown away, "the heat is not working" and "the heat is working" look identical.

Today you fix that. The data is **600 practice reviews of a food court, generated by code for this exercise** (so everyone gets the same data), each labelled positive or negative. The task is called **sentiment analysis**: deciding whether a text is happy or unhappy. You'll:
- count **word pairs**, so "not cold" and "cold" become different columns
- score reviews with a **word list**, first plainly and then with a rule for *not*
- **train** classifiers on single words and on word pairs
- choose the best method fairly, then test it **once**
- read the mistakes it still makes

### Today's five steps
| Part | Step | What you do | Exercise |
|---|---|---|---|
| 1 | Pairs | See why "not cold" and "cold" need different columns | E1 |
| 2 | Word list | Score reviews with a list of happy and unhappy words | E2 |
| 3 | Train and choose | Train Naive Bayes on words, then on pairs; pick the best on validation | E3 |
| 4 | Inside | See which words and pairs the model relies on | E4 |
| 5 | Test once | Score the chosen model on the test reviews and read its mistakes | E5 |

**For every exercise:** predict → write the code → check → explain.

**How to work**
- Run cells from top to bottom with **Shift + Enter**.
- In an exercise cell, **write the code yourself** under each numbered step. Keep the variable names exactly as written; the Check cell looks for them.
- `PASS` means that step is done. `FIX` tells you what to look at.
- If a Check cell says an exercise **is not finished yet**, that is not an error: write the code in the exercise cell above, run it, then run the Check cell again.

In [1]:
import re
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, confusion_matrix

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)
RANDOM_STATE = 42

def check(label, passed):
    """Print PASS or FIX for one requirement."""
    print(("PASS  " if passed else "FIX   ") + label)

def ready(value, step):
    """True when the exercise above is done. Otherwise print a reminder and return False.

    It never raises, so a blank exercise cell can never stop the class with a red
    traceback: the Check cell just tells the student what to do next.
    """
    if value is None:
        print(f"{step} is not finished yet.")
        print("Write your code in the exercise cell above, run that cell, then run this one again.")
        return False
    return True

print("Setup complete.")

Setup complete.


# Part 0 · The reviews, and three piles
| Column | Meaning |
|---|---|
| `review_id` | Review number. Never a model input. |
| `review` | The review text |
| `sentiment` | **The answer:** positive or negative |

**This is practice data**, generated by the code in the next cell so that everyone gets the same 600 reviews. It imitates short app reviews. About 3% of the labels are flipped on purpose, because in real review data the star rating sometimes doesn't match what the person wrote.

**Three piles, not two.** Today you compare five methods and pick one. If you picked using the test reviews, the test score would no longer be an honest estimate: you'd have chosen the method that happened to do best on those exact reviews. So:

| Pile | Size | Used for |
|---|---|---|
| **training** | 360 | learning (the trained models only) |
| **validation** | 120 | comparing the five methods and choosing one |
| **test** | 120 | scoring the chosen method **once**, at the very end |

It is the same rule as choosing `max_depth` with cross-validation in September: choose on data that is not the test set.

In [2]:
rng = np.random.default_rng(1028)

# PRACTICE DATA: 600 food-court reviews generated by this code for today's exercise.
# They imitate short app reviews. Each one is labelled positive or negative.
foods = ["pizza", "burger", "salad", "fries", "coffee", "sandwich", "pasta", "soup", "tacos", "wrap", "noodles", "omelette"]
good = ["good", "great", "delicious", "fresh", "tasty", "hot", "crispy", "excellent"]
bad = ["cold", "bland", "stale", "soggy", "awful", "greasy", "salty", "overcooked"]
staff_good = ["friendly", "fast", "helpful", "nice"]
staff_bad = ["rude", "slow", "unhelpful", "careless"]
when = ["", "", "", " today", " at lunch", " this week", " again", " last night"]
endings = ["", "", "", " Will come back.", " Recommend it.", " Every time.", " Never again.", " So disappointing.", " Not sure why."]

def pick(options):
    return options[rng.integers(len(options))]

def be(food):
    return "were" if food in ("fries", "tacos", "noodles") else "was"

def positive_review():
    f, g, b, s = pick(foods), pick(good), pick(bad), pick(staff_good)
    text = pick([
        f"The {f} {be(f)} {g}{pick(when)}.",
        f"{g.capitalize()} {f} and {s} staff.",
        f"The {f} {be(f)} not {b} at all{pick(when)}.",           # a bad word, made positive
        f"The {f} {be(f)} never {b}, and the staff was {s}.",
        f"Not bad at all, the {f} {be(f)} {g}.",
        f"The line was long but the {f} {be(f)} {g}.",           # mixed, still positive overall
        f"I was not disappointed, the {f} {be(f)} {g}.",
        f"Honestly not {b} like people say{pick(when)}.",
    ])
    return text + pick(endings[:6])

def negative_review():
    f, g, b, s = pick(foods), pick(good), pick(bad), pick(staff_bad)
    text = pick([
        f"The {f} {be(f)} {b}{pick(when)}.",
        f"The {f} {be(f)} not {g} at all{pick(when)}.",           # a good word, made negative
        f"The {f} {be(f)} never {g}, and the staff was {s}.",
        f"Great, another long wait for {b} {f}.",                # sarcasm: "great" in a bad review
        f"The staff is {pick(staff_good)} but the {f} {be(f)} {b}.",  # mixed, still negative overall
        f"I wanted to like it but the {f} {be(f)} not {g}.",
        f"Honestly not {g} like people say{pick(when)}.",
        f"{b.capitalize()} {f} and {s} staff.",
    ])
    return text + pick(endings[:3] + endings[5:])

rows = [{"review": positive_review(), "sentiment": "positive"} for _ in range(330)]
rows += [{"review": negative_review(), "sentiment": "negative"} for _ in range(270)]
reviews = pd.DataFrame(rows)
# About 3% of the labels are flipped ON PURPOSE, to imitate real review data, where the
# star rating sometimes doesn't match what the person wrote.
flip = rng.random(len(reviews)) < 0.03
reviews.loc[flip, "sentiment"] = reviews.loc[flip, "sentiment"].map({"positive": "negative", "negative": "positive"})
reviews = reviews.sample(frac=1, random_state=6).reset_index(drop=True)
reviews.insert(0, "review_id", [f"R{i:03d}" for i in range(1, len(reviews) + 1)])

# First 60% training; the remaining 40% split in half: validation and test.
X_train, X_rest, y_train, y_rest = train_test_split(
    reviews["review"], reviews["sentiment"], test_size=0.4,
    stratify=reviews["sentiment"], random_state=RANDOM_STATE)
X_val, X_test, y_val, y_test = train_test_split(
    X_rest, y_rest, test_size=0.5, stratify=y_rest, random_state=RANDOM_STATE)

print("training:", len(X_train), "| validation:", len(X_val), "| test:", len(X_test))
display(reviews["sentiment"].value_counts())
display(reviews.head(8))

training: 360 | validation: 120 | test: 120


sentiment
positive    329
negative    271
Name: count, dtype: int64

,review_id,review,sentiment
0,R001,The wrap was not cold at all again. Will come back.,positive
1,R002,The line was long but the omelette was tasty. Will come back.,positive
2,R003,"Not bad at all, the omelette was crispy. Every time.",positive
3,R004,"The wrap was never soggy, and the staff was friendly.",positive
4,R005,"I was not disappointed, the soup was delicious.",positive
5,R006,The tacos were not soggy at all.,negative
6,R007,"The tacos were never cold, and the staff was friendly.",positive
7,R008,I wanted to like it but the noodles were not tasty. Every time.,negative


**Write:** read the 8 reviews above. Find one positive review that contains an unhappy word, and one negative review that contains a happy word.

*Your answer:*

# Part 1 · Word pairs (bigrams)
Compare: **"The fries were cold."** and **"The fries were not cold."**

With scikit-learn's English stop-word list, both become just *fries, cold*: *the*, *were* and **not** are all on the list. Even without the list, single words can't tell them apart well: the second one has every word of the first, plus *not*.

The fix: also count **pairs of neighbouring words**, called **bigrams** (two-word *n-grams*).
- "The fries were not cold" gives the pairs: *the fries, fries were, were not, not cold*.
- "The fries were cold" gives: *the fries, fries were, were cold*.

Now **"not cold"** and **"were cold"** are different columns. In scikit-learn it is one setting:

```python
CountVectorizer(ngram_range=(1, 2))   # single words (1) AND pairs (2)
```

**Don't remove stop words blindly.** The standard list contains *not*, *no* and *never*, which flip meaning. Removing *the* or *were* would usually do no harm; removing *not* does. Today we simply keep every word. (A custom list without the negators also works; see the challenge tasks.)

In [3]:
two_reviews = ["The fries were cold.", "The fries were not cold."]

### E1 · Words only, then words and pairs
**Predict:** with `stop_words="english"`, will the two reviews get the same row of counts?
Yes, it will remove the word "not"

In [4]:
words_table = None
pairs_table = None

# Write your code below. Your code must create the variables set to None above.

# 1. words_table: the counts of two_reviews with CountVectorizer(stop_words="english"),
#    as a DataFrame with one row per review and one column per word. Display it.
#    Hint: v = CountVectorizer(stop_words="english");  m = v.fit_transform(two_reviews)
#    Hint: pd.DataFrame(m.toarray(), columns=v.get_feature_names_out(), index=two_reviews)
v = CountVectorizer(stop_words="english")
m = v.fit_transform(two_reviews)
words_table = pd.DataFrame(
    m.toarray(), columns=v.get_feature_names_out(), index=two_reviews
)
display(words_table)
# 2. pairs_table: the same with CountVectorizer(ngram_range=(1, 2)) and NO stop-word list.
#    Display it.
v = CountVectorizer(ngram_range=(1, 2))
m = v.fit_transform(two_reviews)

pairs_table = pd.DataFrame(
    m.toarray(), columns=v.get_feature_names_out(), index=two_reviews
)
display(pairs_table)

,cold,fries
The fries were cold.,1,1
The fries were not cold.,1,1


,cold,fries,fries were,not,not cold,the,the fries,were,were cold,were not
The fries were cold.,1,1,1,0,0,1,1,1,1,0
The fries were not cold.,1,1,1,1,1,1,1,1,0,1


**Check E1**

In [5]:
if ready(words_table, "E1") and ready(pairs_table, "E1"):
    check("each table has one row per review", len(words_table) == 2 and len(pairs_table) == 2)
    check("with stop words removed, 'not' is gone", "not" not in words_table.columns)
    check("pairs_table has the pair 'not cold'", "not cold" in pairs_table.columns)
    check("pairs_table kept the word 'not' (no stop-word list)", "not" in pairs_table.columns)
    check("the two reviews now have different rows", not (pairs_table.iloc[0] == pairs_table.iloc[1]).all())

PASS  each table has one row per review
PASS  with stop words removed, 'not' is gone
PASS  pairs_table has the pair 'not cold'
PASS  pairs_table kept the word 'not' (no stop-word list)
PASS  the two reviews now have different rows


**Write:** are the two rows of `words_table` the same? Which columns of `pairs_table` tell the reviews apart?

*Your answer:*
The two rows in `words_table` are the same. The columns `not_cold`, `were cold`, and `were not` tell the reviews apart. 

# Part 2 · A word list (lexicon) scorer
Before training anything, try the simplest possible method: two hand-made **word lists**.

1. Split the review into words (lowercase).
2. Each word in the happy list scores **+1**; each word in the unhappy list scores **−1**.
3. Add up. Score above 0 → **positive**; 0 or below → **negative**.

The weakness is *not*. So add one rule:

4. **Negation rule:** a word that comes right after *not*, *never* or *no* counts the **opposite** (+1 becomes −1, −1 becomes +1).

Slides 12–16 do it by hand on two reviews. Word lists need no training, so they don't use the training pile; we score them on the **validation** reviews, like every method we compare today.

In [6]:
HAPPY = {"good", "great", "delicious", "fresh", "tasty", "hot", "crispy", "excellent",
         "friendly", "fast", "helpful", "nice", "love", "worth", "recommend"}
UNHAPPY = {"bad", "cold", "bland", "stale", "soggy", "awful", "greasy", "salty", "overcooked",
           "rude", "slow", "unhelpful", "careless", "long", "disappointing", "disappointed"}
NEGATIONS = {"not", "never", "no"}


def words_of(text):
    """Lowercase words of a review, without punctuation."""
    return re.findall(r"[a-z']+", text.lower())


print(words_of("Great, another long wait for cold fries."))

['great', 'another', 'long', 'wait', 'for', 'cold', 'fries']


### E2 · Score the validation reviews with the word lists
**Predict:** the word lists were written by a person who knows English. Will they do better than always saying "positive"?

Yes, they should 

In [7]:
list_accuracy = None
rule_accuracy = None

# Write your code below. Your code must create the variables set to None above.


# 1. Write a function list_score(text): +1 for every word in HAPPY, -1 for every word in UNHAPPY.
#    Hint: w = words_of(text);  sum(x in HAPPY for x in w) - sum(x in UNHAPPY for x in w)
def list_score(text):
    w = words_of(text)
    return sum(x in HAPPY for x in w) - sum(x in UNHAPPY for x in w)


# 2. Write a function rule_score(text): the same, but a word right after a word in NEGATIONS
#    counts the opposite.
#    Hint: loop over the words with a variable flip = False.
#          value = (x in HAPPY) - (x in UNHAPPY);  if flip: value = -value
#          add value to the score, then set flip = x in NEGATIONS for the next word.
def rule_score(text):
    score = 0
    flip = None
    for x in words_of(text):
        value = (x in HAPPY) - (x in UNHAPPY)
        if flip:
            value = -value
        score += value
        flip = x in NEGATIONS
    return score


# 3. Predict every review in X_val: "positive" if the score > 0, otherwise "negative".
#    Hint: np.where(X_val.map(list_score) > 0, "positive", "negative")
#    list_accuracy and rule_accuracy: accuracy_score(y_val, predictions) for each.
#    Print both, and the scores of "The fries were not crispy at all." with each function.
list_score_predictions = np.where(X_val.map(list_score) > 0, "positive", "negative")
list_accuracy = accuracy_score(y_val, list_score_predictions)

rule_score_predictions = np.where(X_val.map(rule_score) > 0, "positive", "negative")
rule_accuracy = accuracy_score(y_val, rule_score_predictions)

print(list_accuracy)
print(rule_accuracy)


0.525
0.9083333333333333


**Check E2**

In [8]:
if ready(list_accuracy, "E2") and ready(rule_accuracy, "E2"):
    check("list_score counts happy minus unhappy words", list_score("good fresh cold") == 1)
    check("list_score ignores 'not'", list_score("not crispy") == 1)
    check("rule_score flips the word after 'not'", rule_score("not crispy") == -1)
    check("rule_score flips after 'never' too", rule_score("never cold") == 1)
    check("rule_score only flips the NEXT word", rule_score("not crispy but tasty") == 0)
    check("both accuracies are on the 120 validation reviews",
          np.isclose(rule_accuracy, (np.where(X_val.map(rule_score) > 0, "positive", "negative") == y_val).mean()))

PASS  list_score counts happy minus unhappy words
PASS  list_score ignores 'not'
PASS  rule_score flips the word after 'not'
PASS  rule_score flips after 'never' too
PASS  rule_score only flips the NEXT word
PASS  both accuracies are on the 120 validation reviews


**Write:** compare the two accuracies. What did the negation rule fix? Find one kind of review it still gets wrong.

*Your answer:*
The negation rule fixes the issue where words that should be negative were marked as positive. But, that only negates the next word, not the statement. 

# Part 3 · Let the data choose the words, then choose a method
A trained classifier learns its own "word list" from the **training** reviews, with a weight for every word, and for every **pair** if you count pairs. Same Naive Bayes as Friday; only the vectorizer changes:

```python
make_pipeline(CountVectorizer(), MultinomialNB())                      # single words
make_pipeline(CountVectorizer(ngram_range=(1, 2)), MultinomialNB())    # words and pairs
```

No stop-word list in either: we keep *not*.

**The baseline, done properly.** "Always predict the most common class" must take the most common class from the **training** labels (that's all you'd know in advance), then score that prediction on the validation reviews.

### E3 · Compare five methods on the validation reviews
**Predict:** will Naive Bayes on single words beat the hand-made word lists with the "not" rule?

In [9]:
words_model = None
pairs_model = None
words_accuracy = None
pairs_accuracy = None
baseline = None

# Write your code below. Your code must create the variables set to None above.

# 1. words_model: a pipeline of CountVectorizer() and MultinomialNB(), fitted on X_train, y_train.
#    pairs_model: the same with CountVectorizer(ngram_range=(1, 2)).
words_model = make_pipeline(CountVectorizer(), MultinomialNB()).fit(X_train, y_train)
pairs_model = make_pipeline(CountVectorizer(ngram_range=(1, 2)), MultinomialNB()).fit(
    X_train, y_train
)

# 2. words_accuracy and pairs_accuracy on the VALIDATION reviews (X_val, y_val).
wpred = words_model.predict(X_val)
words_accuracy = accuracy_score(y_val, wpred)
print(words_accuracy)

ppred = pairs_model.predict(X_val)
pairs_accuracy = accuracy_score(y_val, ppred)
print(pairs_accuracy)
# 3. baseline: take the most common class in y_train, predict it for every validation review,
#    and score it on y_val.
#    Hint: majority = y_train.value_counts().idxmax();  baseline = (y_val == majority).mean()
majority = y_train.value_counts().idxmax()
baseline = (y_val == majority).mean()
# 4. Display one table of all five methods (baseline, word lists, word lists + not rule,
#    words model, pairs model) with their validation accuracy.
models = [
    ("Baseline", baseline),
    ("Word List", list_accuracy),
    ("Word lists+ not rule", rule_accuracy),
    ("Words model", words_accuracy),
    ("Pairs model", pairs_accuracy),
]

accuracies = pd.DataFrame(models, columns=["Model", "Accuracy"])
display(accuracies.sort_values(by="Accuracy", ascending=False))

0.8666666666666667
0.9333333333333333


,Model,Accuracy
4,Pairs model,0.933333
2,Word lists+ not rule,0.908333
3,Words model,0.866667
0,Baseline,0.550000
1,Word List,0.525000


**Check E3**

In [10]:
if ready(words_model, "E3") and ready(pairs_model, "E3") and ready(pairs_accuracy, "E3") and ready(baseline, "E3"):
    check("the words model counts single words only", words_model[0].ngram_range == (1, 1))
    check("the pairs model counts words and pairs", pairs_model[0].ngram_range == (1, 2))
    check("neither model removes stop words", words_model[0].stop_words is None and pairs_model[0].stop_words is None)
    check("both were trained on the training reviews only",
          len(pairs_model[0].vocabulary_) == len(CountVectorizer(ngram_range=(1, 2)).fit(X_train).vocabulary_))
    check("accuracies are on the validation reviews",
          np.isclose(pairs_accuracy, (pairs_model.predict(X_val) == y_val).mean())
          and np.isclose(words_accuracy, (words_model.predict(X_val) == y_val).mean()))
    check("the baseline's class comes from the training labels",
          np.isclose(baseline, (y_val == y_train.value_counts().idxmax()).mean()))

PASS  the words model counts single words only
PASS  the pairs model counts words and pairs
PASS  neither model removes stop words
PASS  both were trained on the training reviews only
PASS  accuracies are on the validation reviews
PASS  the baseline's class comes from the training labels


**Write:** compare the five methods. What happened, and why? Which method would you choose for the final test?

*Your answer:*
Out of all the models, the pairs model had the highest accuracy of 93.33...%. I would chose this model for the final test.  

# Part 4 · What did the pairs model learn?
For every word or pair, Naive Bayes stores its chance in positive reviews and its chance in negative reviews (`feature_log_prob_`, as logs). Their **difference** says which way a feature pushes:

`push = log(chance in positive) − log(chance in negative)`

- push > 0: the feature is more common in positive reviews, so it pushes towards positive. A push of 2 means about e² ≈ 7 times more likely in positive reviews.
- push < 0: it pushes towards negative.
- push ≈ 0: it says nothing.

`model.classes_` tells you which row is which: `["negative", "positive"]`, alphabetical, so row 1 is positive.

Slides 22–24 work it out by hand for the pair *not bad*: 23 times in positive training reviews, 0 times in negative ones, push 3.05.

### E4 · The strongest features, and four reviews of your own
**Predict:** will the strongest negative features be single words, or pairs?

In [11]:
push = None
my_proba = None

my_reviews = [
    "The fries were not cold.",
    "The fries were cold.",
    "Great, another long wait for cold fries.",
    "The pasta was not bad but the staff was rude.",
]

# Write your code below. Your code must create the variables set to None above.

# 1. push: a pandas Series {feature: push} for the pairs model, sorted from most negative to most positive.
#    Hint: vec, nb = pairs_model[0], pairs_model[1]
#    Hint: pd.Series(nb.feature_log_prob_[1] - nb.feature_log_prob_[0], index=vec.get_feature_names_out())
#    Display the 10 most negative and the 10 most positive.
vec, nb = pairs_model[0], pairs_model[1]
push = pd.Series(
    nb.feature_log_prob_[1] - nb.feature_log_prob_[0], index=vec.get_feature_names_out()
).sort_values(ascending=True)
display(push.head(10), push.tail(10))
# 2. my_proba: the pairs model's probabilities for my_reviews, one row per review,
#    one column per sentiment. Display it, rounded to 2 decimals.
#    Hint: pd.DataFrame(pairs_model.predict_proba(my_reviews), index=my_reviews, columns=pairs_model.classes_)
my_proba = pd.DataFrame(
    pairs_model.predict_proba(my_reviews),
    index=my_reviews,
    columns=pairs_model.classes_,
)

never again     -3.074780
staff is        -2.963554
is              -2.963554
for             -2.902930
another         -2.902930
another long    -2.902930
wait for        -2.902930
wait            -2.902930
great another   -2.902930
long wait       -2.902930
dtype: float64

was hot     2.508716
all the     3.047713
not bad     3.047713
bad at      3.047713
bad         3.047713
line was    3.396019
the line    3.396019
long but    3.396019
was long    3.396019
line        3.396019
dtype: float64

**Check E4**

In [12]:
if ready(push, "E4") and ready(my_proba, "E4"):
    check("push has one value per feature of the pairs model", len(push) == len(pairs_model[0].vocabulary_))
    check("push is sorted from most negative to most positive", push.is_monotonic_increasing)
    check("push is positive minus negative (row 1 minus row 0)",
          np.isclose(push.max(), (pairs_model[1].feature_log_prob_[1] - pairs_model[1].feature_log_prob_[0]).max()))
    check("my_proba has one row per review and two columns", my_proba.shape == (len(my_reviews), 2))
    check("each row adds up to 1", np.allclose(my_proba.sum(axis=1), 1))

PASS  push has one value per feature of the pairs model
PASS  push is sorted from most negative to most positive
PASS  push is positive minus negative (row 1 minus row 0)
PASS  my_proba has one row per review and two columns
PASS  each row adds up to 1


**Write:** (a) How many of the 10 strongest features on each side are pairs? Which make sense, and which look like an accident of how these reviews were written? (b) Look at the four reviews. Which results surprise you? Explain each surprise.

*Your answer:*
(a) **Negative side:** 6 of 10 are pairs (`never again`, `staff is`, `another long`, `wait for`, `great another`, `long wait`). **Positive side:** 8 of 10 are pairs (`was hot`, `all the`, `not bad`, `bad at`, `line was`, `the line`, `long but`, `was long`).
- *Make sense:* Genuine sentiment pairs like `never again`, `long wait`, and `not bad`.
- *Accidents of review generation:* Features like `line`, `the line`, `was long`, and even `bad` are strongly positive because they exclusively came from fixed positive templates (`"The line was long but..."`, `"Not bad at all..."`). Similarly, `staff is`, `great another`, `for`, and `another` are strongly negative because of the sarcastic/wait templates.

(b) **Surprises in the four reviews:**
- *"The pasta was not bad but the staff was rude.":* The model predicts positive despite "rude" because `not bad`, `bad at`, and `bad` are among the strongest positive features.
- *"Great, another long wait for cold fries.":* Correctly predicted negative despite starting with "Great", because sarcastic pairs `great another`, `another long`, and `long wait` heavily weigh negative.
- *"The fries were not cold." vs. "The fries were cold.":* The pairs model successfully distinguishes them because the bigram `not cold` prevents `cold` from driving a negative prediction.

# Part 5 · Test once, then read the mistakes
On the validation reviews you chose the pairs model. Now, and **only now**, score it on the test reviews: that is its honest final number, because nothing was chosen using those reviews.

Then read the reviews it gets wrong, and sort them into reasons, as on Friday.

The model also reports a **probability** for its prediction (`predict_proba`). Call it the model's probability, not "how sure it is": as you saw in E4, Naive Bayes probabilities tend to be too extreme.

### E5 · The final test, and the mistakes
**Predict:** will the test accuracy be higher or lower than the validation accuracy? What kinds of review will it get wrong?

In [13]:
test_accuracy = None
mistakes = None

# Write your code below. Your code must create the variables set to None above.

# 1. test_accuracy: the pairs model's accuracy on X_test, y_test. Print it.
#    (Don't change anything after seeing it: that would be choosing with the test set.)
test_accuracy = pairs_model.score(X_test, y_test)
# 2. mistakes: the test reviews it gets wrong, with columns review, true, predicted,
#    and probability (the model's probability for the class it predicted).
#    Hint: pred = pairs_model.predict(X_test);  wrong = pred != y_test.values
#    Hint: pairs_model.predict_proba(X_test).max(axis=1)
#    Display it.
pred = pairs_model.predict(X_test)
wrong = pred != y_test.values 

mistakes = pd.DataFrame({
    "review": X_test[wrong],
    "true": y_test[wrong],
    "predicted": pred[wrong],
    "probability": pairs_model.predict_proba(X_test).max(axis=1)[wrong]
})
display(mistakes)

# 3. Print the confusion matrix on the test reviews.
#    Hint: confusion_matrix(y_test, pred, labels=["negative", "positive"])
display(confusion_matrix(y_test, pred, labels=["negative", "positive"]))

,review,true,predicted,probability
128,The burger was cold this week.,negative,positive,0.992007
130,The sandwich was not stale at all at lunch.,negative,positive,0.990188
443,Honestly not stale like people say again.,positive,negative,0.594557
186,The omelette was not crispy at all last night. Every time.,negative,positive,0.673297
563,The burger was greasy.,negative,positive,0.732187
169,Honestly not salty like people say today.,positive,negative,0.844326
179,Honestly not overcooked like people say last night.,positive,negative,0.831369
194,The burger was not crispy at all.,negative,positive,0.802992


array([[49,  5],
       [ 3, 63]])

**Check E5**

In [14]:
if ready(test_accuracy, "E5") and ready(mistakes, "E5"):
    check("test_accuracy is the pairs model on the test reviews",
          np.isclose(test_accuracy, (pairs_model.predict(X_test) == y_test).mean()))
    check("one row per wrong test prediction", len(mistakes) == int((pairs_model.predict(X_test) != y_test.values).sum()))
    check("every row really is wrong", (mistakes["true"] != mistakes["predicted"]).all())
    check("mistakes has review, true, predicted and probability",
          {"review", "true", "predicted", "probability"} <= set(mistakes.columns))
    check("probability is at least 0.5 (the probability of the predicted class)",
          bool((mistakes["probability"] >= 0.5).all()))

PASS  test_accuracy is the pairs model on the test reviews
PASS  one row per wrong test prediction
PASS  every row really is wrong
PASS  mistakes has review, true, predicted and probability
PASS  probability is at least 0.5 (the probability of the predicted class)


**Write:** report the final test accuracy. Sort the mistakes into reasons. Which could a better model or more data fix, and which are not the model's fault?

*Your answer:*
- **Final test accuracy:** **93.33%** (112/120 correct; confusion matrix: 49 true negative, 63 true positive, 5 false positive, 3 false negative).

- **Mistakes breakdown:**
  1. **Flipped ground-truth labels (Not the model's fault):**
     - Reviews like *"The burger was cold this week."* (true: `negative`, pred: `positive`), *"The sandwich was not stale at all at lunch."* (true: `negative`, pred: `positive`), and *"The burger was greasy."* (true: `negative`, pred: `positive`).
     - The data generator intentionally flipped ~3% of labels as label noise. The model actually made the semantically correct prediction; the ground truth was mislabeled.
  2. **Complex / longer phrasing unseen patterns (Fixable with more data / better model):**
     - *"Honestly not salty/stale/overcooked like people say..."* (true: `positive`, pred: `negative`). The bigrams `like people`, `people say`, and the negative words outweighed the positive negation because `honestly not` is not a strong enough bigram compared to the negative adjectives.
     - *"The omelette was not crispy at all..."* (true: `negative`, pred: `positive`). Sarcastic/negated-good expressions where "not crispy" couldn't overcome the positive weight of "crispy" across distant words without higher-order n-grams or contextual transformers (e.g., BERT).

### The AI's analysis
The food court manager asks an AI assistant to set up sentiment analysis. It replies:

> "Sentiment is easy: just count positive and negative words from a standard list; there is no need to train anything. Before counting, always clean the text by removing stop words. And if you do want a trained model, your word-pair model is 93% accurate, which proves it understands what students mean, including sarcasm."

**Write:** find **three** problems in the AI's answer. For each one, give the evidence from your notebook.

*Your answer:*
1. **"Always clean the text by removing stop words":** Standard stop-word lists remove crucial negation words like `"not"`, `"no"`, and `"never"`. In Part 1 (E1), removing stop words causes *"The fries were cold"* and *"The fries were not cold"* to produce identical feature vectors (`['cold', 'fries']`), completely destroying the meaning.
2. **"Just count positive and negative words from a standard list; no need to train anything":** The untrained word-list baseline achieved only ~52.5% accuracy on validation (worse than the 55% majority-class baseline), failing on mixed, negated, or context-dependent reviews. Training a model with bigrams significantly improved performance to over 90%.
3. **"93% accurate proves it understands what students mean, including sarcasm":** The model does not "understand" meaning or sarcasm. It relies on superficial n-gram correlations from template artifacts. For example, in E4, phrases like *"great another"*, *"line was"*, and even the word *"bad"* became strong sentiment signals solely because fixed synthetic sentence structures paired them with specific labels, not because the model understands linguistic sarcasm.

### Challenge tasks (optional)
1. Try `ngram_range=(1, 3)` (words, pairs and triples) and compare it **on the validation reviews**. Does it beat the pairs model? (Don't use the test set for this.)
2. Make a stop-word list without the negators and use it with `ngram_range=(1, 2)`:
   `keep = {"not", "no", "never"}`
   `my_stop = [w for w in CountVectorizer(stop_words="english").get_stop_words() if w not in keep]`
   What happens to validation accuracy?
3. Add `"but"` handling to `rule_score`: only count the words **after** "but". Does it fix the mixed reviews on validation?

# Exit ticket
Answer without running more code.
1. What is a bigram? Give the bigrams of "the soup was not hot".
2. Why must stop-word removal be done carefully for sentiment? Which words must you keep?
3. Why did we compare the five methods on validation reviews and use the test reviews only once?

*Your answers:*
1. A bigram is a pair of adjacent words in a sequence of text. For "the soup was not hot", the bigrams are: `"the soup"`, `"soup was"`, `"was not"`, and `"not hot"`.
2. Standard stop-word lists remove negators and polarity shifters, which completely reverses or eliminates the sentiment. You must keep negation words like `"not"`, `"no"`, and `"never"`.
3. To prevent data leakage / overfitting to the evaluation set. Using validation to pick the best method ensures the final evaluation on the test set is an honest, unbiased estimate of generalization performance on unseen data.

# Submit
- Restart the kernel and run all cells. Every check should print `PASS`.
- Every *Your answer* and *Your prediction* cell is filled in.
- Submit this notebook with its outputs.
- **Assignment 3 is due tonight, Wednesday October 28, 11:59 PM.**